In [1]:
# Import necessary libraries
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import os
import dagshub

In [2]:
# setup mlflow tracking 
mlflow.set_tracking_uri("https://dagshub.com/yogibaba7/mlops_mini_project.mlflow")
dagshub.init(repo_owner='yogibaba7', repo_name='mlops_mini_project', mlflow=True)



Accessing as yogibaba7

Initialized MLflow to track repo "yogibaba7/mlops_mini_project"

Repository yogibaba7/mlops_mini_project initialized!

In [3]:
# Load the data
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [4]:

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

# Normalize the text data
df = normalize_text(df)


In [5]:
x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})

In [6]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['content'])
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
# Define hyperparameter grid for Logistic Regression
param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

In [8]:
# Set the experiment name
mlflow.set_experiment("LoR Hyperparameter Tuning")

2026/04/25 18:04:53 INFO mlflow.tracking.fluent: Experiment with name 'LoR Hyperparameter Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/b8b260ad5b5d43a08fe3d20cc7a9aea5', creation_time=1777120492866, experiment_id='3', last_update_time=1777120492866, lifecycle_stage='active', name='LoR Hyperparameter Tuning', tags={}>

In [9]:
# Start the parent run for hyperparameter tuning
with mlflow.start_run():

    # Perform grid search
    grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5, scoring='f1', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    # log each parameter combination as a child run
    for i in range(len(grid_search.cv_results_['params'])):
        params = grid_search.cv_results_['params'][i]
        mean_score = grid_search.cv_results_['mean_test_score'][i]
        std_score = grid_search.cv_results_['std_test_score'][i]

        with mlflow.start_run(run_name=f"LR with params: {params}", nested=True):

            model = LogisticRegression(**params)
            model.fit(X_train, y_train)
            
            # Model evaluation
            y_pred = model.predict(X_test)
            accuracy = accuracy_score(y_test, y_pred)
            precision = precision_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            
            # Log parameters and metrics
            mlflow.log_params(params)
            mlflow.log_metric("mean_cv_score", mean_score)
            mlflow.log_metric("std_cv_score", std_score)
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("f1_score", f1)
            
            
            # Print the results for verification
            print(f"Mean CV Score: {mean_score}, Std CV Score: {std_score}")
            print(f"Accuracy: {accuracy}")
            print(f"Precision: {precision}")
            print(f"Recall: {recall}")
            print(f"F1 Score: {f1}")

    # Log the best run details in the parent run
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    mlflow.log_params(best_params)
    mlflow.log_metric("best_f1_score", best_score)
    
    print(f"Best Params: {best_params}")
    print(f"Best F1 Score: {best_score}")

    # Save and log the notebook
    mlflow.log_artifact("exp3_logr_tfidf_hp.ipynb")

    # Log model
    mlflow.sklearn.log_model(grid_search.best_estimator_, "model")

Mean CV Score: 0.5399203902306713, Std CV Score: 0.025226018708197986
Accuracy: 0.6824096385542169
Precision: 0.7966666666666666
Recall: 0.470935960591133
F1 Score: 0.5919504643962848
🏃 View run LR with params: {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/yogibaba7/mlops_mini_project.mlflow/#/experiments/3/runs/2f1e8e82ed70432a84db4e154c213ba5
🧪 View experiment at: https://dagshub.com/yogibaba7/mlops_mini_project.mlflow/#/experiments/3
Mean CV Score: 0.7788343876777836, Std CV Score: 0.005412341479432073
Accuracy: 0.7773493975903615
Precision: 0.763584366062917
Recall: 0.7891625615763547
F1 Score: 0.7761627906976745
🏃 View run LR with params: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/yogibaba7/mlops_mini_project.mlflow/#/experiments/3/runs/41f9b74e57a14e35975eb1ebfd129768
🧪 View experiment at: https://dagshub.com/yogibaba7/mlops_mini_project.mlflow/#/experiments/3
Mean CV Score: 0.7786127587611203, Std CV Score: 0.014830784

c:\Users\Yogesh\anaconda3\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Mean CV Score: 0.7835189147794264, Std CV Score: 0.00818603860013079
Accuracy: 0.7879518072289157
Precision: 0.7659574468085106
Recall: 0.8157635467980295
F1 Score: 0.7900763358778626
🏃 View run LR with params: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/yogibaba7/mlops_mini_project.mlflow/#/experiments/3/runs/218af709704441ffab4c771103c79f04
🧪 View experiment at: https://dagshub.com/yogibaba7/mlops_mini_project.mlflow/#/experiments/3
Mean CV Score: 0.792483408564114, Std CV Score: 0.007769692248220801
Accuracy: 0.7937349397590362
Precision: 0.7792578496669839
Recall: 0.8068965517241379
F1 Score: 0.7928363988383349
🏃 View run LR with params: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/yogibaba7/mlops_mini_project.mlflow/#/experiments/3/runs/4016b55ccb71444fa51b2850f6d49a7e
🧪 View experiment at: https://dagshub.com/yogibaba7/mlops_mini_project.mlflow/#/experiments/3
Best Params: {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}

2026/04/25 18:12:24 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run dapper-quail-21 at: https://dagshub.com/yogibaba7/mlops_mini_project.mlflow/#/experiments/3/runs/98ecaee6207e472db08f1f3ebea737fd
🧪 View experiment at: https://dagshub.com/yogibaba7/mlops_mini_project.mlflow/#/experiments/3
